# 0. Library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# 1. Function definitions

In [2]:
def f_monthly(
        mortgage_total, 
        annual_repayment_rate, 
        annual_interest_rate, 
        months, 
        monthly_avgift=0,
        plot=True,
        title='Monthly payment & balance',
        show_payment=True,
        show_balance=True
        ):
    """Return per-month lists (length `months`) instead of only the last month.

    The repayment is fixed each month, computed from the original mortgage amount;
    interest is still charged on the outstanding balance.
    """
    monthly_interest_rate = annual_interest_rate/12
    monthly_repayment_rate = annual_repayment_rate/12
    fixed_repayment = mortgage_total * monthly_repayment_rate  # based on original amount
    mortgage_paid = 0
    paid, remain, repayment, interest, avgift, total = [], [], [], [], [], []
    
    for m in np.arange(months):
        mortgage_remain = mortgage_total - mortgage_paid
        monthly_repayment = min(fixed_repayment, mortgage_remain)
        monthly_interest = mortgage_remain * monthly_interest_rate
        monthly_total = monthly_repayment + monthly_interest + monthly_avgift
        mortgage_paid += monthly_repayment
        paid.append(mortgage_paid)                    # cumulative, end of month
        remain.append(mortgage_total - mortgage_paid) # end of month
        repayment.append(monthly_repayment)
        interest.append(monthly_interest)
        avgift.append(monthly_avgift)
        total.append(monthly_total)
    
    if plot:
        months_axis = np.arange(1, months+1)
        hover = '%{y:,.2~f}<extra></extra>'

        fig = go.Figure()
        layout = dict(
            title=title, xaxis_title='month', hovermode='x unified',
            legend=dict(groupclick='toggleitem', x=1.12, xanchor='left', y=1),
            margin=dict(l=60, r=160, t=40, b=50)
            )

        if show_payment:
            fig.add_trace(go.Scatter(
                x=months_axis, y=repayment, name='repayment', hovertemplate=hover,
                legendgroup='payment', legendgrouptitle_text='Left axis (payment)'
                ))
            fig.add_trace(go.Scatter(x=months_axis, y=interest, name='interest', hovertemplate=hover, legendgroup='payment'))
            fig.add_trace(go.Scatter(x=months_axis, y=avgift, name='avgift', hovertemplate=hover, legendgroup='payment'))
            fig.add_trace(go.Scatter(x=months_axis, y=total, name='total', hovertemplate=hover, legendgroup='payment'))
            layout['yaxis'] = dict(title='SEK / month')

        if show_balance:
            fig.add_trace(go.Scatter(
                x=months_axis, y=paid, name='paid', yaxis='y2', line=dict(dash='dash'), hovertemplate=hover,
                legendgroup='balance', legendgrouptitle_text='Right axis (balance)'
                ))
            fig.add_trace(go.Scatter(
                x=months_axis, y=remain, name='remaining', yaxis='y2', line=dict(dash='dash'), hovertemplate=hover,
                legendgroup='balance'
                ))
            layout['yaxis2'] = dict(title='SEK balance', overlaying='y', side='right')

        fig.update_layout(**layout)
        fig.show()

    return paid, remain, repayment, interest, avgift, total

def f_compound(
        base,
        year_plan,
        plot=False,
        title='Cumulative savings'
        ):
    """Return per-year lists of cumulative savings. Both the number of years and the
    growth rate are derived from `year_plan` rather than passed separately.

    Each year, interest is credited on the current balance, then that year's net
    addition/subtraction is applied.

    `year_plan` is a list of segment dicts describing additions or subtractions over
    an inclusive, 1-indexed year range:
        {
            'year': [start, end],            # inclusive year range, e.g. [1, 5];
                                               # a single year may be given as 10 or
                                               # [10] instead of [10, 10]
            'yearly_addition': 12000,         # amount added each year in range;
                                               # negative = withdrawal. Default 0.
            'yearly_addition_change': 0.02,   # fractional growth of yearly_addition
                                               # each year within the range. Default 0.
            'yearly_growth_rate': 0.09,       # growth rate for years in this range.
                                               # Optional: carried forward from the
                                               # last year it was set (0 before any
                                               # segment sets it).
        }
    `years` (the length of the simulation) is the highest year covered by any
    segment. Years not covered by any segment get no addition. Segments may
    overlap: their additions are summed (so a lump sum is just a one-year-range
    segment layered on a longer recurring plan); if segments disagree on the growth
    rate for an overlapping year, the last one in the list wins.
    """
    if not year_plan:
        raise ValueError('year_plan must contain at least one segment')

    normalized = []
    years = 0
    for segment in year_plan:
        segment_year = segment['year']
        if isinstance(segment_year, (list, tuple)):
            y_start, y_end = segment_year if len(segment_year) == 2 else (segment_year[0], segment_year[0])
        else:
            y_start = y_end = segment_year
        normalized.append((y_start, y_end, segment))
        years = max(years, y_end)

    additions = np.zeros(years)
    rates = np.full(years, np.nan)

    for y_start, y_end, segment in normalized:
        addition = segment.get('yearly_addition', 0)
        addition_change = segment.get('yearly_addition_change', 0)
        rate_override = segment.get('yearly_growth_rate')

        for year in range(max(y_start, 1), y_end + 1):
            idx = year - 1
            additions[idx] += addition
            if rate_override is not None:
                rates[idx] = rate_override
            addition *= (1 + addition_change)

    last_rate = 0.0
    for idx in range(years):
        if np.isnan(rates[idx]):
            rates[idx] = last_rate
        else:
            last_rate = rates[idx]

    balance = base
    contributed = base
    contributed_list, interest_list, addition_list, total = [], [], [], []

    for idx in np.arange(years):
        year_interest = balance * rates[idx]
        balance += year_interest + additions[idx]
        contributed += additions[idx]
        contributed_list.append(contributed)          # cumulative, end of year
        interest_list.append(balance - contributed)   # cumulative, end of year
        addition_list.append(additions[idx])
        total.append(balance)                          # cumulative, end of year

    if plot:
        years_axis = np.arange(1, years+1)
        hover = '%{y:,.2~f}<extra></extra>'

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=years_axis, y=contributed_list, name='contributed', stackgroup='one', hovertemplate=hover))
        fig.add_trace(go.Scatter(x=years_axis, y=interest_list, name='interest', stackgroup='one', hovertemplate=hover))
        fig.add_trace(go.Scatter(x=years_axis, y=total, name='total', line=dict(dash='dot', color='black'), hovertemplate=hover))
        fig.update_layout(
            title=title, xaxis_title='year', yaxis_title='SEK', hovermode='x unified',
            margin=dict(l=60, r=20, t=40, b=50)
            )
        fig.show()

    return contributed_list, interest_list, addition_list, total

# 2. Housing payments estimator

In [14]:
AVGIFT = 3958+6700/12 # per month
MORTGAGE = 3.4e6-3.4e5
REPAYMENT_RATE = 0.02 # per year
INTEREST_RATE = 0.03 # per year
MONTHS = 12*30

paid, remain, monthly_repayment, monthly_interest, monthly_avgift, monthly_total = f_monthly(
    mortgage_total=MORTGAGE, 
    annual_repayment_rate=REPAYMENT_RATE,
    annual_interest_rate=INTEREST_RATE, 
    months=MONTHS, 
    monthly_avgift=AVGIFT, 
    plot=True,
    title='Monthly payment & balance (Årsta flat)'
    )

In [12]:
AVGIFT = 5312+6200/12 # per month
MORTGAGE = 3.3e6-3.3e5
REPAYMENT_RATE = 0.02 # per year
INTEREST_RATE = 0.03 # per year
MONTHS = 12*30

paid, remain, monthly_repayment, monthly_interest, monthly_avgift, monthly_total = f_monthly(
    mortgage_total=MORTGAGE, 
    annual_repayment_rate=REPAYMENT_RATE,
    annual_interest_rate=INTEREST_RATE, 
    months=MONTHS, 
    monthly_avgift=AVGIFT, 
    plot=True,
    title='Monthly payment & balance (Farsta flat)'
    )

# 3. Cumulative savings estimator

In [9]:
BASE = 1.2e5+4e5
ADDITION_GROWTH_RATE = 0.015 # yearly addition increases % per year
YEARLY_RATE = 0.085 # per year

YEAR_PLAN = [
    # saving X / month
    {
        'year': [1, 15], 
        'yearly_addition': 12*12000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
        }, 
    # saving X / month
    {
        'year': [16, 20], 
        'yearly_addition': 12*14000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
        }, 
    # lump sum withdrawal
    {
        'year': 20, 
        'yearly_addition': -1.5e6
        },
    # no addition, subtraction only
    {
        'year': [21, 35], 
        'yearly_addition': -12*25000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
    },
    # # no addition, no subtraction
    # {
    #     'year': [36, 40], 
    #     'yearly_addition': 0, 
    #     'yearly_addition_change': 0, 
    #     'yearly_growth_rate': YEARLY_RATE
    # },
    # no addition, no subtraction
    {
        'year': [36, 50], 
        'yearly_addition': -12*35000*2, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
    },
]

contributed, interest_earned, yearly_addition, total = f_compound(
    base=BASE,
    year_plan=YEAR_PLAN,
    plot=True,
    title='Cumulative savings (Y)'
    )

In [10]:
BASE = 4e4+4e4
ADDITION_GROWTH_RATE = 0.0 # yearly addition increases % per year
YEARLY_RATE = 0.085 # per year

YEAR_PLAN = [
    # saving X / month
    {
        'year': [1, 20], 
        'yearly_addition': 12*6000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
        }, 
    # lump sum withdrawal
    # {
    #     'year': 20, 
    #     'yearly_addition': -1.5e6
    #     },
    # no addition, subtraction only
    {
        'year': [21, 55], 
        'yearly_addition': -12*25000, 
        'yearly_addition_change': ADDITION_GROWTH_RATE, 
        'yearly_growth_rate': YEARLY_RATE
    },
    # no addition, no subtraction
    # {
    #     'year': [36, 40], 
    #     'yearly_addition': 0, 
    #     'yearly_addition_change': 0, 
    #     'yearly_growth_rate': YEARLY_RATE
    # },
]

contributed, interest_earned, yearly_addition, total = f_compound(
    base=BASE,
    year_plan=YEAR_PLAN,
    plot=True,
    title='Cumulative savings (A)'
    )